# Embedding
OpenAI의 텍스트 임베딩은 텍스트 문자열의 관련성을 측정합니다. 임베딩은 일반적으로 다음 용도로 사용됩니다.

- 두 텍스트 사이의 관련성을 측정하는 데 사용할 수 있는 텍스트의 숫자 표현  
  
임베딩은 부동 소수점 숫자로 표시되는 벡터입니다. 두 벡터 사이의 거리는 관련성 을 측정합니다. 거리가 작을수록 관련성이 높음을 나타내고, 거리가 멀면 관련성이 낮다는 것을 나타냅니다.  

<img src=https://cdn.sanity.io/images/vr8gru94/production/e016bbd4d7d57ff27e261adf1e254d2d3c609aac-2447x849.png width=600 />

In [1]:
# OpenAI 임베딩(text-embedding-3-small) 모델을 사용한 문장 임베딩 예제
import numpy as np
import pandas as pd
from openai import OpenAI
import os
from sklearn.metrics.pairwise import cosine_similarity

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv()) # read local .env file

True

In [2]:
# OpenAI API 키를 환경변수에서 가져와 클라이언트 생성
from openai import OpenAI 
client = OpenAI()  # OpenAI 클라이언트 생성

embedding_model = "text-embedding-3-small"  # 사용할 텍스트 임베딩 모델

# 샘플 데이터: 간단한 문장들의 모음
sentences = [
    '나는 인공지능 공부를 좋아한다.',
    '인공지능은 매우 흥미롭다.',
    '오늘 날씨가 흐리고 비가 온다.'
]

In [3]:
# OpenAI Embedding API를 사용하여 문장 임베딩 함수
def get_embeddings(texts, model='text-embedding-3-small'):
    response = client.embeddings.create(
        input=texts,
        model=model
    )
    embeddings = [data.embedding for data in response.data]
    return np.array(embeddings)

In [4]:
# 문장 임베딩 실행
embeddings = get_embeddings(sentences)
print(f'임베딩 차원: {embeddings.shape[1]}')
print(f'임베딩 결과 shape: {embeddings.shape}')

임베딩 차원: 1536
임베딩 결과 shape: (3, 1536)


In [5]:
# 임베딩 결과 확인
print('첫 번째 문장의 임베딩 벡터 (처음 10개 값):')
print(embeddings[0][:10])

첫 번째 문장의 임베딩 벡터 (처음 10개 값):
[-0.02177429 -0.01200867 -0.01317596  0.01151276  0.04919434 -0.01535034
 -0.01948547  0.07971191 -0.02648926 -0.01010132]


In [6]:
# 임베딩 결과를 DataFrame으로 변환하여 시각화
embedding_df = pd.DataFrame(embeddings, index=sentences)
print(embedding_df.head())

                       0         1         2         3         4         5     \
나는 인공지능 공부를 좋아한다. -0.021774 -0.012009 -0.013176  0.011513  0.049194 -0.015350   
인공지능은 매우 흥미롭다.    -0.015343  0.010643 -0.042297  0.030060  0.010666 -0.035217   
오늘 날씨가 흐리고 비가 온다.  0.019867 -0.001801 -0.074280 -0.024399  0.029694  0.045837   

                       6         7         8         9     ...      1526  \
나는 인공지능 공부를 좋아한다. -0.019485  0.079712 -0.026489 -0.010101  ...  0.004887   
인공지능은 매우 흥미롭다.    -0.028671  0.067566  0.020157  0.009102  ... -0.002491   
오늘 날씨가 흐리고 비가 온다.  0.029343  0.040863 -0.052612 -0.038879  ... -0.013519   

                       1527      1528      1529      1530      1531      1532  \
나는 인공지능 공부를 좋아한다.  0.026428 -0.067200  0.024109  0.018265  0.026703  0.017578   
인공지능은 매우 흥미롭다.     0.023087 -0.031052  0.011276  0.016449  0.005489  0.038483   
오늘 날씨가 흐리고 비가 온다. -0.021942 -0.023926 -0.008461  0.024078  0.036346 -0.010300   

                       1533      1534      15

In [7]:
# 문장 간 코사인 유사도 계산
similarity_matrix = cosine_similarity(embeddings)

print('\n문장 간 코사인 유사도:')
for i in range(len(sentences)):
    for j in range(i+1, len(sentences)):
        similarity = similarity_matrix[i][j]
        print(f'{sentences[i]} vs {sentences[j]}: {similarity:.4f}') 


문장 간 코사인 유사도:
나는 인공지능 공부를 좋아한다. vs 인공지능은 매우 흥미롭다.: 0.5846
나는 인공지능 공부를 좋아한다. vs 오늘 날씨가 흐리고 비가 온다.: 0.1016
인공지능은 매우 흥미롭다. vs 오늘 날씨가 흐리고 비가 온다.: 0.1661
